# Phase 4 - train the LSTM on Colab

This notebook is orchestration only. It mounts Drive, puts the workspace on the
path, copies the dataset to local disk, and calls the `ml` command line. No
training logic and no metric arithmetic lives here - if you find yourself
adding either, it belongs in `ml/src/ml/` where it can be tested.

**Before running:** put `data/training/` (65 MB) and `data/dataset/lives.parquet`
in a Drive folder. The dataset is gitignored and cannot be committed.

In [ ]:
REPO = "https://github.com/YOUR_USER/ai-predictive-iot-rag-demo.git"
DRIVE_DATA = "/content/drive/MyDrive/pdm/data"
DRIVE_RUNS = "/content/drive/MyDrive/pdm/runs"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!git clone --depth 1 $REPO /content/repo
%cd /content/repo

# Not `pip install -e`. The workspace pins `requires-python = ">=3.12,<3.13"`
# and Colab now runs 3.13, so pip refuses to build it. Both packages are pure
# Python, so putting `src` on the path is exactly what an editable install
# would do, and it skips the version check entirely.
#
# `%env` rather than `sys.path`: the `!python -m ml ...` cells below run in a
# subshell and only inherit environment variables.
%env PYTHONPATH=/content/repo/ml/src:/content/repo/services/simulator/src

# Colab ships numpy, pyarrow and torch. Checked rather than assumed.
import numpy, pyarrow
print(f"numpy {numpy.__version__} | pyarrow {pyarrow.__version__}")

## Copy the dataset off Drive

Training reads the signals array many times per epoch. A memory-mapped read
over the Drive FUSE mount turns each one into network round trips, so the
artifact is copied local first. It is 65 MB - memory is not a concern.

In [ ]:
!mkdir -p /content/data
!cp -r $DRIVE_DATA/training /content/data/training
!ls -la /content/data/training

In [ ]:
import time

RUN = f"run-{time.strftime('%Y%m%d-%H%M%S')}"
OUT = f"{DRIVE_RUNS}/{RUN}"
!mkdir -p $OUT

# Checkpoints go straight to Drive so a disconnect costs one epoch, and
# --resume picks up from the last one.
!python -m ml model train \
  --config ml/configs/lstm.json \
  --artifact /content/data/training \
  --out $OUT

In [ ]:
!python -m ml model verify --run $OUT
!python -m ml model predict --run $OUT --artifact /content/data/training --split test
!python -m ml model predict --run $OUT --artifact /content/data/training --split test_shift

In [ ]:
# Evaluation needs no GPU and no torch - it reads the prediction tables.
for split in ("test", "test_shift"):
    !python -m ml model evaluate \
      --predictions $OUT/predictions_{split}.parquet \
      --artifact /content/data/training \
      --out $OUT/eval_{split}

In [ ]:
!cat $OUT/eval_test/report.txt

## Reading the results

The report prints the clock baseline on the same rows as the model, and the
renderer refuses to do otherwise. A model that has not cleared the baseline by
a wide margin has measured a clock. `test_shift` holds machines whose lives are
6 or 48 hours - a model that learned timing collapses there.

Run at least two seeds before quoting a margin. The baseline is a single
deterministic number, so one run above it cannot be distinguished from a lucky
initialisation.